In [1]:
!pip install pyspark

In [2]:
import csv
import time
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, HTML
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg

DATASET_PATH = './assets/synthetic_sales_data_400k.csv'

In [3]:
with open(DATASET_PATH, newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f'No. of Data Rows: {len(rows)}')
for row in rows[:5]:
    print(row)

No. of Data Rows: 400000
{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '2130.42', 'Customer_Type': 

In [4]:
# Sequential Filtering
start = time.time()
filtered = [row for row in rows if float(row['Sales_Amount']) > 1000]
filter_time = time.time() - start
print(f'Filtered rows  : {len(filtered)}')
print(f'Filtering took : {filter_time:.4f}s')
for row in filtered[:5]:
    print(row)

Filtered rows  : 396390
Filtering took : 0.4937s
{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '213

In [5]:
# Sequential Aggregation (Total Sales Amount per Product_Category)
start = time.time()
aggregated = {}
for row in rows:
    cat = row['Product_Category']
    aggregated[cat] = aggregated.get(cat, 0) + float(row['Sales_Amount'])
agg_time = time.time() - start
print(f'Grouping took  : {agg_time:.4f}s')
for cat, total in aggregated.items():
    print(f'{cat}: {total:.2f}')

Grouping took  : 0.9506s
Food: 7131972595.98
Electronics: 7109945168.08
Clothing: 7138801521.35
Furniture: 7133332433.30


In [6]:
# Sequential Sorting (by Sale_Date ascending)
start = time.time()
sorted_rows = sorted(rows, key=lambda r: r['Sale_Date'])
sort_time = time.time() - start
print(f'Sorting took   : {sort_time:.4f}s')
for row in sorted_rows[:5]:
    print(row)

Sorting took   : 0.7782s
{'Product_ID': '1419', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '18217.64', 'Quantity_Sold': '6', 'Product_Category': 'Clothing', 'Unit_Cost': '2179.2', 'Unit_Price': '3264.81', 'Customer_Type': 'New', 'Discount': '0.07', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'West-Charlie'}
{'Product_ID': '1410', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Bob', 'Region': 'North', 'Sales_Amount': '66543.25', 'Quantity_Sold': '16', 'Product_Category': 'Food', 'Unit_Cost': '4528.61', 'Unit_Price': '5134.51', 'Customer_Type': 'Returning', 'Discount': '0.19', 'Payment_Method': 'Cash', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Bob'}
{'Product_ID': '1456', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '25519.87', 'Quantity_Sold': '34', 'Product_Category': 'Clothing', 'Unit_Cost': '626.63', 'Unit_Price': '807.08', 'Customer_Type': 'Returning', 

In [7]:
def filter_chunk(chunk):
    return [row for row in chunk if float(row['Sales_Amount']) > 1000]

def agg_chunk(chunk):
    result = {}
    for row in chunk:
        cat = row['Product_Category']
        result[cat] = result.get(cat, 0) + float(row['Sales_Amount'])
    return result

def sort_chunk(chunk):
    return sorted(chunk, key=lambda r: r['Sale_Date'])

def parallel_operation(data, func):
    num_workers = multiprocessing.cpu_count()
    size = len(data) // num_workers
    chunks = [data[i*size:(i+1)*size] for i in range(num_workers)]
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(func, chunks))
    return results

In [8]:
# Parallel Filtering
start = time.time()
filtered_chunks = parallel_operation(rows, filter_chunk)
filtered_parallel = [row for chunk in filtered_chunks for row in chunk]
filter_time_p = time.time() - start
print(f'Filtered rows  : {len(filtered_parallel)}')
print(f'Filtering took : {filter_time_p:.4f}s')
for row in filtered_parallel[:5]:
    print(row)

Filtered rows  : 396390
Filtering took : 0.4623s
{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '213

In [9]:
# Parallel Aggregation
start = time.time()
agg_chunks = parallel_operation(rows, agg_chunk)
agg_parallel = {}
for chunk_result in agg_chunks:
    for cat, total in chunk_result.items():
        agg_parallel[cat] = agg_parallel.get(cat, 0) + total
agg_time_p = time.time() - start
print(f'Grouping took  : {agg_time_p:.4f}s')
for cat, total in agg_parallel.items():
    print(f'{cat}: {total:.2f}')

Grouping took  : 0.6590s
Food: 7131972595.98
Electronics: 7109945168.08
Clothing: 7138801521.35
Furniture: 7133332433.30


In [10]:
# Parallel Sorting
start = time.time()
sorted_chunks = parallel_operation(rows, sort_chunk)
sorted_parallel = [row for chunk in sorted_chunks for row in chunk]
sort_time_p = time.time() - start
print(f'Total sorted rows: {len(sorted_parallel)}')
print(f'Sorting took   : {sort_time_p:.4f}s')
for row in sorted_parallel[:5]:
    print(row)

Total sorted rows: 400000
Sorting took   : 0.8174s
{'Product_ID': '1419', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '18217.64', 'Quantity_Sold': '6', 'Product_Category': 'Clothing', 'Unit_Cost': '2179.2', 'Unit_Price': '3264.81', 'Customer_Type': 'New', 'Discount': '0.07', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'West-Charlie'}
{'Product_ID': '1410', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Bob', 'Region': 'North', 'Sales_Amount': '66543.25', 'Quantity_Sold': '16', 'Product_Category': 'Food', 'Unit_Cost': '4528.61', 'Unit_Price': '5134.51', 'Customer_Type': 'Returning', 'Discount': '0.19', 'Payment_Method': 'Cash', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Bob'}
{'Product_ID': '1456', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '25519.87', 'Quantity_Sold': '34', 'Product_Category': 'Clothing', 'Unit_Cost': '626.63', 'Unit_Price': '807.08', 'Cus